In [2]:
import matplotlib as mpl
from matplotlib import pyplot as plt
import seaborn as sns
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
from scipy.io import loadmat
from scipy.signal import stft
import os
import sklearn
from sklearn.preprocessing import MinMaxScaler,StandardScaler
from collections import Counter

from numpy import *
from sklearn.model_selection import train_test_split
import scipy.optimize as optimize
import sklearn.datasets as ds
from sklearn import datasets
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.svm import SVC, LinearSVC
from time import time
from sklearn.preprocessing import label_binarize
from sklearn.multiclass import OneVsRestClassifier
from sklearn.multiclass import OneVsOneClassifier
from sklearn.metrics import roc_curve,auc
from sklearn.model_selection import cross_val_score
from sklearn.metrics import precision_recall_fscore_support
#from sklearn.metrics import plot_confusion_matrix, accuracy_score, f1_score, recall_score, precision_score
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, f1_score, recall_score, precision_score


In [4]:
def init_classifiers():
    '''
    Initialize our machine learning classifier --- 
    where catboost and NN (neural network classification) are not initialized, 
    and most hyperparameters will take default values
            
    '''
    
    model_names = ['SVM','LR','KNN','GBDT','DT','AdaB', 'RF','XGB','LGB','Catboost','NN']

    # the training parameters of each model
    param_grid_svc = [{}]
    param_grid_logistic = [{'C':[0.1], 'penalty':['l1','l2']}]
    param_grid_knn = [{},{'n_neighbors':list(range(3,8))}]
    param_grid_gbdt = [{}]
    param_grid_tree = [{}]
    param_grid_boost = [{}]
    param_grid_rf = [{}]
    param_grid_xgb = [{'tree_method':['gpu_hist'], 'gpu_id':[0]}]
    #param_grid_xgb = [{}]
    param_grid_lgb = [{}]

    return([#(SVC(), model_names[0], param_grid_svc), 
#             (LogisticRegression(), model_names[1], param_grid_logistic),
#             (KNeighborsClassifier(), model_names[2], param_grid_knn),
#             (GradientBoostingClassifier(), model_names[3], param_grid_gbdt),
#             (DecisionTreeClassifier(), model_names[4], param_grid_tree),
#             (AdaBoostClassifier(), model_names[5], param_grid_boost),
            (RandomForestClassifier(), model_names[6], param_grid_rf),
             #(xgb.XGBClassifier(), model_names[7], param_grid_xgb),
#             (lgb.sklearn.LGBMClassifier(), model_names[8], param_grid_lgb)
           ])

def model_evaluation_dict(train_x, train_y, test_x_1, test_y_1,model, model_name, params):
    '''
    Perform 10 fold crossvalidation, fit model with train data and evaluate its performance 
    return performance dict
            
    '''
    
    clf = GridSearchCV(model, params, cv=10)
    
    X_train, X_test, y_train, y_test = train_x, test_x_1, train_y, test_y_1
    clf.fit(X_train, y_train)
    params = clf.best_params_
    
    Training_score = clf.score(X_train,y_train)
    Score_10_10_10 = clf.score(X_test, y_test) 
    cvres = clf.cv_results_ 
    cvscore = cvres['mean_test_score'][clf.best_index_] 
    macro_precision, macro_recall, macro_f1_score, macro_support=\
    precision_recall_fscore_support(y_test, clf.predict(X_test), average='macro') 
    micro_precision, micro_recall, micro_f1_score, micro_support=\
    precision_recall_fscore_support(y_test, clf.predict(X_test), average='micro') 
    if not params:
        # empty params dict
        params = 'default'
    # return a dictionary
    d_info = {'Classifier': model_name,'param':params,'Traing score': Training_score,'Score non bias set': Score_10_10_10, \
         'CV Score':cvscore, \
         'Precision(Macro)': macro_precision, 'Precision(Micro)': micro_precision,\
         'Recall(Macro)': macro_recall, 'Recall(Micro)': micro_recall,\
         'F1 Score(Macro)': macro_f1_score, 'F1 Score(Micro)': micro_f1_score}
    
    confusion_1 = ConfusionMatrix( labels=['Focused','Unfocused','Drowsed'])
    confusion_1.update(clf.predict(X_test), y_test)
#     confusion_2 = ConfusionMatrix( labels=['Focused','Unfocused','Drowsed'])
#     confusion_2.update(clf.predict(X_test), y_test)
#     confusion_3 = ConfusionMatrix(num_classes=2, labels=['Undrowsed','Drowsed'])
#     confusion_3.update(clf.predict(test_x_3), test_y_3)

    
    return clf,d_info,confusion_1


#def set_seed(seed=42):
    '''
    set all random seed to 42

    '''   
#     random.seed(seed) # python
#    np.random.seed(seed) # numpy
#    torch.manual_seed(seed) # pytorch
#    torch.cuda.manual_seed(seed)
#    torch.backends.cudnn.deterministic = True
#    torch.backends.cudnn.benchmark = False